## 1. Installs and Imports

In [ ]:
!pip install -U torch torchvision tqdm pandas scipy Pillow datasets transformers

In [ ]:
!pip install --force-reinstall --no-cache-dir scipy
!pip uninstall -y Pillow
!pip install Pillow
!pip install numpy==1.26.4

In [ ]:
from transformers import CLIPProcessor, CLIPModel, CLIPVisionModel
from datasets import load_dataset, load_from_disk
from tqdm import tqdm
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader
import numpy as np
import pandas as pd
import copy
from collections import defaultdict

## 1. Model Prepping

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"

In [ ]:
class VisionHooks(torch.nn.Module):
    def __init__(self, vision):
        super().__init__()
        self.clip = vision
        self.cls_tokens = []
    
    def forward(self, images):
        self.cls_tokens = []

        # From line 724 in tranformers/src/transformers/models/clip/modeling_clip.py
        x = self.clip.embeddings(images)
        x = self.clip.pre_layrnorm(x)

        # Adds the embedding layer output before transformer layers
        # self.cls_tokens = [x[:, 0, :]]

        # Dummy ones because I don't need these for visual encoder
        batch_size, seq_len, _ = x.shape
        attention_mask = torch.ones((batch_size, seq_len), dtype=torch.bool, device=x.device)
        attention_mask = attention_mask[:, None, None, :] # [B, 1, 1, 5]
        causal_attention_mask = None  # Not used for vision

        # Here's where the transformer layers are. Line # 552. Only input_embeds is needed. 
        # Replacing the following line
        # x = self.clip.encoder(x)
        for block in self.clip.encoder.layers:
            layer_outputs = block(
                x, # hidden states
                attention_mask,
                causal_attention_mask,
                output_attentions=False,
            )
            x = layer_outputs[0] # Batch Size, Sequence Length, Hidden Size
            self.cls_tokens.append(x[:, 0, :]) # (Batch_Size, Hidden_Size) -> (64, 768)
        
        return {
            "cls": [i for i in self.cls_tokens] 
        }

In [ ]:
class AugmentedCLIP(torch.nn.Module):
    def __init__(self, full, vision=None, W=None, b=None, transform_stage=-1):
        super().__init__()
        self.vision = vision if vision is not None else full.vision_model
        self.text = full.text_model
        self.visual_projection = full.visual_projection
        self.text_projection = full.text_projection
        self.logit_scale = full.logit_scale
        self.W = torch.from_numpy(W.astype(np.float32)).to(device) if W is not None else None
        self.b = torch.from_numpy(b.astype(np.float32)).to(device) if b is not None else None
        self.transform_stage = transform_stage
    
    def get_image_features(self, pixel_values):
        pixel_values = pixel_values.to(device)
        x = self.vision.embeddings(pixel_values)
        x = self.vision.pre_layrnorm(x)

        batch_size, seq_len, _ = x.shape
        attention_mask = torch.ones((batch_size, seq_len), dtype=torch.bool, device=x.device)
        attention_mask = attention_mask[:, None, None, :] # [B, 1, 1, 5]
        causal_attention_mask = None  # Not used for vision
        for i, block in enumerate(self.vision.encoder.layers):
            layer_outputs = block(
                x, # hidden states
                attention_mask,
                causal_attention_mask,
                output_attentions=False,
            )
            x = layer_outputs[0] # Batch Size, Sequence Length, Hidden Size
            if i == self.transform_stage:
                cls = x[:, 0, :].to(torch.float32)
                if self.W is None:
                    self.W = torch.eye(cls.shape[-1], device=cls.device, dtype=cls.dtype)
                if self.b is None:
                    self.b = torch.zeros(cls.shape[-1], device=cls.device, dtype=cls.dtype)
                cls = cls @ self.W + self.b
                x[:, 0, :] = cls
                break
        
        raw_cls = x[:, 0, :]
        pooled = self.vision.post_layernorm(x[:, 0, :])

        return raw_cls, pooled # raw cls token, normalized cls token
        
    def forward(self, pixel_values, text_embeds):
        raw_cls, vision_outputs = self.get_image_features(pixel_values)
        image_embeds = self.visual_projection(vision_outputs) # 768 -> 512 Dimensions
        image_embeds = image_embeds / image_embeds.norm(dim=1, keepdim=True)

        logits = image_embeds @ text_embeds.T
        logits = logits * self.logit_scale.exp()

        return logits, raw_cls

In [ ]:
processer = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")

dataset = load_dataset("ylecun/mnist", cache_dir="/workspace/.hf_cache")
split = dataset["train"].train_test_split(test_size=0.2, seed=66)

train = split["train"]
val = split["test"]
test = dataset["test"]

def transform_example(batch):
    labels = [f"a photo of a {label}"for label in batch["label"]]
    images = batch["image"]

    processed = processer(text=labels, images=images, return_tensors="pt", padding=True)
    batch["input_ids"] = processed["input_ids"].numpy()
    batch["attention_mask"] = processed["attention_mask"].numpy()
    batch["pixel_values"] = processed["pixel_values"].numpy()

    return batch

train = train.map(transform_example, batched=True, batch_size=64)
val = val.map(transform_example, batched=True, batch_size=64)
test = test.map(transform_example, batched=True, batch_size=64)

In [ ]:
train.save_to_disk("/workspace/preprocessed/MNIST/train_converted")
val.save_to_disk("/workspace/preprocessed/MNIST/val_converted")
test.save_to_disk("/workspace/preprocessed/MNIST/test_converted")

In [ ]:
train = load_from_disk("/workspace/preprocessed/MNIST/train_converted")
val = load_from_disk("/workspace/preprocessed/MNIST/val_converted")
test = load_from_disk("/workspace/preprocessed/MNIST/test_converted")

In [ ]:
print(train.column_names)

In [ ]:
train.set_format(type="torch", columns=["label", "input_ids", "attention_mask", "pixel_values"])
val.set_format(type="torch", columns=["label", "input_ids", "attention_mask", "pixel_values"])
test.set_format(type="torch", columns=["label", "input_ids", "attention_mask", "pixel_values"])

def clip_collate_fn(batch):
    pixel_values = torch.stack([example["pixel_values"] for example in batch])
    input_ids = torch.stack([example["input_ids"] for example in batch])
    attention_mask = torch.stack([example["attention_mask"] for example in batch])
    labels = torch.tensor([example["label"] for example in batch], dtype=torch.long)
    
    return {
        "pixel_values": pixel_values,
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "labels": labels
    }

train_loader = DataLoader(train, batch_size=64, shuffle=True, collate_fn=clip_collate_fn, pin_memory=True, num_workers=8, persistent_workers=True)
val_loader = DataLoader(val, batch_size=64, shuffle=False, collate_fn=clip_collate_fn, pin_memory=True, num_workers=8, persistent_workers=True)
test_loader = DataLoader(test, batch_size=64, shuffle=False, collate_fn=clip_collate_fn, pin_memory=True, num_workers=8, persistent_workers=True)

## 3. Extracting Embeddings

In [ ]:
def leastSquares(Z0, Z1):
    W_full, residuals, rank, s = np.linalg.lstsq(Z0, Z1, rcond=None) 
    return W_full, residuals

def cosineSimilarity(aug_cls, fine_cls):
    eps = 1e-8
    out_aug_cls = F.normalize(aug_cls, dim=1, eps=eps)
    out_fine_cls = F.normalize(fine_cls, dim=1, eps=eps)
    co_sim_cls = (out_aug_cls * out_fine_cls).sum(dim=1).mean().item()
    return co_sim_cls

In [ ]:
f_t_vision = CLIPVisionModel.from_pretrained('tanganke/clip-vit-base-patch32_mnist')
refer = CLIPModel.from_pretrained("openai/clip-vit-base-patch32")
refer = refer.float().to(device)

base_copy = copy.deepcopy(refer)
base_H = VisionHooks(base_copy.vision_model)
base_H = base_H.eval().to(device)

fine_tuned = copy.deepcopy(refer)
fine_tuned.vision_model.load_state_dict(f_t_vision.vision_model.state_dict())
fine_tuned_H = VisionHooks(fine_tuned.vision_model)
fine_tuned_H = fine_tuned_H.eval().to(device)

In [ ]:
# Extracting Embeddings
Z0 = {}
for i in range(12): 
    Z0[i] = []

Z1_twelfth_lsr = []

with torch.no_grad():
    for batch in tqdm(train_loader, desc=f"Extracting Train Dataset Vectors"):
        images = batch["pixel_values"].to(device, non_blocking=True)

        out_base = base_H(images)
        out_fine_tuned = fine_tuned_H(images)
        
        for i in range(12):
            Z0[i].append(out_base["cls"][i].float().cpu())

        Z1_twelfth_lsr.append(out_fine_tuned["cls"][11].float().cpu())

Z1_twelfth_lsr = torch.cat(Z1_twelfth_lsr)
Z1_twelfth_lsr = Z1_twelfth_lsr.cpu().numpy()

W = {}
resid = {}

for key, value in Z0.items():
    value = torch.cat(value)
    value = value.cpu().numpy()
    W[key], resid[key] = leastSquares(value, Z1_twelfth_lsr)

# Augmenting Models
aug = {}

for i in range(12): 
    model = AugmentedCLIP(base_copy, base_copy.vision_model, W=W[i], transform_stage=i)
    model = model.eval().to(device)
    aug[i] = model

## 3. Evaluating

In [ ]:
base = AugmentedCLIP(copy.deepcopy(refer))
base = base.eval().to(device)

f_t = copy.deepcopy(refer)
f_t.vision_model.load_state_dict(f_t_vision.vision_model.state_dict())
fine_tuned = AugmentedCLIP(f_t, f_t.vision_model)
fine_tuned = fine_tuned.eval().to(device)

In [ ]:
label_texts = [f"a photo of a {i}" for i in range(10)]
text_inputs = processer(text=label_texts, return_tensors="pt", padding=True)
text_inputs = {k: v.to(device) for k, v in text_inputs.items()}
with torch.no_grad():
    text_outputs = refer.text_model(
        input_ids=text_inputs["input_ids"],
        attention_mask=text_inputs["attention_mask"]
    )
    text_embeds = refer.text_projection(text_outputs.pooler_output)
    text_embeds = text_embeds / text_embeds.norm(dim=1, keepdim=True) # Shape of (10, 768)

In [ ]:
correct_base = 0
correct_fine_tuned = 0
total_samples = 0

correct = {}
sim_cls = {}
for i in range(12):
    correct[i] = 0
    sim_cls[i] = []

with torch.no_grad():
    for batch in tqdm(test_loader, desc="Evaluating"):
        images = batch["pixel_values"].to(device, non_blocking=True)
        labels = batch["labels"].to(device, non_blocking=True)
        total_samples += labels.size(0)

        # Base Model
        logits_base, raw_cls_base = base(images, text_embeds)
        preds = logits_base.argmax(dim=1)
        correct_base += (preds == labels).sum().item()

        # Fine-Tuned Model
        logits_fine_tuned, raw_cls_fine_tuned = fine_tuned(images, text_embeds)
        preds = logits_fine_tuned.argmax(dim=1)
        correct_fine_tuned += (preds == labels).sum().item()

        # Augmented Base Models
        for i in range(12):
            logits_augmented, raw_cls_augmented = aug[i](images, text_embeds)
            preds = logits_augmented.argmax(dim=1)
            correct[i] += (preds == labels).sum().item()
            sim_cls[i].append(cosineSimilarity(raw_cls_fine_tuned, raw_cls_augmented))

In [ ]:
for i in range(12):
    correct[i] = correct[i] / total_samples
    sim_cls[i] = np.mean(sim_cls[i])
base_acc = correct_base / total_samples
fine_tuned_acc = correct_fine_tuned / total_samples

print(f"Base Accuracy: {base_acc}")
print(f"Fine-Tuned Accuracy: {fine_tuned_acc}")
for i in range(12):
    print(f"\tAugmented {i+1} - Last (12) Layer Accuracy: {correct[i]}")
    print(f"\tAverage Cosine Similarity of CLS Token of Augmented {i+1} Layer to Fine-Tuned Last (12) Layer: {sim_cls[i]:.4f}")

In [ ]:
# Not Right. Base and Fine-Tuned should be 0.475 and 0.995 respectively
# Base Accuracy: 0.2504
# Fine-Tuned Accuracy: 0.7165
# 	Augmented 1 - Last (12) Layer Accuracy: 0.6204
# 	Average Cosine Similarity of CLS Token of Augmented 1 Layer to Fine-Tuned Last (12) Layer: 0.8945
# 	Augmented 2 - Last (12) Layer Accuracy: 0.6256
# 	Average Cosine Similarity of CLS Token of Augmented 2 Layer to Fine-Tuned Last (12) Layer: 0.8966
# 	Augmented 3 - Last (12) Layer Accuracy: 0.6422
# 	Average Cosine Similarity of CLS Token of Augmented 3 Layer to Fine-Tuned Last (12) Layer: 0.9027
# 	Augmented 4 - Last (12) Layer Accuracy: 0.6643
# 	Average Cosine Similarity of CLS Token of Augmented 4 Layer to Fine-Tuned Last (12) Layer: 0.9203
# 	Augmented 5 - Last (12) Layer Accuracy: 0.6845
# 	Average Cosine Similarity of CLS Token of Augmented 5 Layer to Fine-Tuned Last (12) Layer: 0.9382
# 	Augmented 6 - Last (12) Layer Accuracy: 0.6957
# 	Average Cosine Similarity of CLS Token of Augmented 6 Layer to Fine-Tuned Last (12) Layer: 0.9477
# 	Augmented 7 - Last (12) Layer Accuracy: 0.7026
# 	Average Cosine Similarity of CLS Token of Augmented 7 Layer to Fine-Tuned Last (12) Layer: 0.9557
# 	Augmented 8 - Last (12) Layer Accuracy: 0.715
# 	Average Cosine Similarity of CLS Token of Augmented 8 Layer to Fine-Tuned Last (12) Layer: 0.9653
# 	Augmented 9 - Last (12) Layer Accuracy: 0.7172
# 	Average Cosine Similarity of CLS Token of Augmented 9 Layer to Fine-Tuned Last (12) Layer: 0.9689
# 	Augmented 10 - Last (12) Layer Accuracy: 0.7214
# 	Average Cosine Similarity of CLS Token of Augmented 10 Layer to Fine-Tuned Last (12) Layer: 0.9684
# 	Augmented 11 - Last (12) Layer Accuracy: 0.7196
# 	Average Cosine Similarity of CLS Token of Augmented 11 Layer to Fine-Tuned Last (12) Layer: 0.9659
# 	Augmented 12 - Last (12) Layer Accuracy: 0.7182
# 	Average Cosine Similarity of CLS Token of Augmented 12 Layer to Fine-Tuned Last (12) Layer: 0.9652